In [1]:
import sys
print(sys.executable)

c:\Users\Hp\AppData\Local\Programs\Python\Python313\python.exe


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import datetime
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
df = pd.read_csv("live_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%d-%m-%y %H:%M:%S")


features = ['cpu_usage', 'gpu_wrk_util', 'avg_mem', 'max_mem', 
            'avg_gpu_wrk_mem', 'max_gpu_wrk_mem', 'read', 'write', 
            'read_count', 'write_count']

scaler = MinMaxScaler()
original_count = len(df)
df = df[df['timestamp'].dt.year >= 2025].copy()


In [ ]:
unique_servers = df['machine'].unique()
start_time = df["timestamp"].iloc[0]
while True:
    end_time = start_time + datetime.timedelta(minutes=10) 
    filtered_logs = df[
        (df["timestamp"] >= start_time) &
        (df["timestamp"] < end_time)
    ]
    for mac in unique_servers:
        server_data = df[df['machine']==mac].copy()
        
        min_time = server_data['timestamp'].min()
        max_time = server_data['timestamp'].max()
        time_gap = max_time - min_time
        
        print(f"First log: {min_time}")
        print(f"Last log:  {max_time}")
        print(f"Total time span: {time_gap}")
        
        # If the time span is larger than a few days for a 3000 log test, it's a bomb.
        if time_gap.days > 2:
            print("🚨 ERROR: MASSIVE TIME GAP DETECTED. Stopping script to save RAM!")
            sys.exit()

        # 3. Safe Resampling
        server_data.set_index('timestamp', inplace=True)
        server_minute = server_data.resample('1min').mean(numeric_only=True)
        
        # Only forward-fill up to 5 minutes to prevent infinite RAM usage
        server_minute.ffill(limit=5, inplace=True) 
        
        print(f"Resampled to {len(server_minute)} clean 1-minute rows.")
        
        # 4. Scale and sequence
        if len(server_minute) > TIME_STEPS:
            data_values = server_minute[features].values
            data_scaled = scaler.fit_transform(data_values)
            
            for i in range(len(data_scaled) - TIME_STEPS):
                sequence = data_scaled[i:(i + TIME_STEPS)]
                all_sequences.append(sequence)
        else:
            print(f"Not enough data for {mac} to create a 10-minute window. Skipping.")
    model = Sequential()
    model.add(LSTM(16, activation='relu', input_shape=(10, 10))) 

    # The Bridge (Holds the compressed data)
    model.add(RepeatVector(10)) 

    # The Decoder (Tries to unpack it and redraw the flashcard)
    model.add(LSTM(16, activation='relu', return_sequences=True))
    model.add(TimeDistributed(Dense(10)))

    # STEP 2: COMPILE IT
    # We tell it how to learn. 'adam' is the standard engine. 
    # 'mse' (Mean Squared Error) tells it to measure how badly it failed at redrawing the data.
    model.compile(optimizer='adam', loss='mse')

    # STEP 3: TRAIN IT
    # We feed it your clean 3,000 logs. 
    # Notice X_train is there twice! It is learning to copy X_train exactly.
    model.fit(X_train, X_train, epochs=10, batch_size=16)

    # STEP 4: PREDICT (Detect Anomalies)
    # If we feed it a crash later, it will try to redraw it, fail, and the 'loss' will be huge!
    predictions = model.predict(X_test)
    start_time =start_time + datetime.timedelta(minutes=1)
    



Fetching data from Kafka...
